In [ ]:
import pandas as pd

# Carregar dados do novo path (já balanceado)
df = pd.read_csv("../data/processed/all_datasets_aligned_balanced.csv")
print(df.head())          # Ver as primeiras linhas
print(df['flag'].value_counts())  # Quantos exemplos tem de cada classe
print(f"Dataset shape: {df.shape}")

In [ ]:
import pandas as pd

# Defina o número de amostras desejado para "BENIGN"
n_samples_benign = 1000_000  # ou 1_000_000, como preferir

# Separe a classe majoritária
benign_df = df[df['flag'] == 'BENIGN']

# Separe as demais classes (minoritárias)
minor_classes_df = df[df['flag'] != 'BENIGN']

# Amostragem aleatória da classe BENIGN
benign_sampled = benign_df.sample(n=n_samples_benign, random_state=42)

# Combine novamente o DataFrame balanceado
df_balanced = pd.concat([benign_sampled, minor_classes_df], axis=0)

# Embaralhar o DataFrame final
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Verificar a nova distribuição
print(df_balanced['flag'].value_counts())

df = df_balanced
#df.to_csv("all_datasets_aligned_balanced.csv", index=False)
del df_balanced, benign_df, minor_classes_df, benign_sampled

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
df = df.dropna()

# Remove a coluna 'id' se existir
if 'id' in df.columns:
    df = df.drop(columns=['id'])
else:
    print("Coluna 'id' não encontrada, continuando...")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Codificar rótulos em números
le = LabelEncoder()
df['flag'] = le.fit_transform(df['flag'])

X = df.drop(columns=['flag'])  # Features
y = df['flag']                 # Rótulos

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

del df  # Liberar memória

In [ ]:
# Verificar tipos de dados e converter colunas problematicas
print("Tipos de dados antes da conversão:")
print(X_train.dtypes)
print()

# Converter colunas que estão como string para float
for col in ['val3', 'val4', 'val6', 'val7']:
    if col in X_train.columns:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
        print(f"Convertido {col} para float")

# Remover possíveis NaN criados na conversão
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_test.mean())

print("\nTipos de dados após conversão:")
print(X_train.dtypes)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import json
import os

# SVM com normalização e GridSearch
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42))
])

param_grid_svm = {
    'svc__kernel': ['rbf', 'linear'],
    'svc__C': [1, 10],
    'svc__gamma': ['scale', 'auto']
}

svm_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid_svm,
    scoring='accuracy',
    cv=svm_cv,
    n_jobs=-1,
    verbose=3
)

# Limitar o tamanho do conjunto de treino se for muito grande
if X_train.shape[0] > 200_000:
    X_train_subset, _, y_train_subset, _ = train_test_split(
        X_train, y_train, train_size=200_000, stratify=y_train, random_state=42
    )
    svm_search.fit(X_train_subset, y_train_subset)
    print(f"Treinado com subconjunto de {X_train_subset.shape[0]} amostras")
else:
    svm_search.fit(X_train, y_train)

print('Melhores parâmetros SVM:')
print(svm_search.best_params_)
print(f"Melhor acurácia SVM (CV): {svm_search.best_score_:.4f}")

svm_best = svm_search.best_estimator_
svm_pred = svm_best.predict(X_test)
print('Relatório de classificação SVM:')
print(classification_report(y_test, svm_pred))

# Salvar resultados em disco
results_dir = os.path.abspath(os.path.join('..', 'results'))
metrics_dir = os.path.join(results_dir, 'metrics', 'svm')
model_dir = os.path.join(results_dir, 'models')
os.makedirs(metrics_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

report = classification_report(y_test, svm_pred)
with open(os.path.join(metrics_dir, 'svm_classification_report.txt'), 'w', encoding='utf-8') as f:
    f.write(report)

cm = confusion_matrix(y_test, svm_pred)
pd.DataFrame(cm).to_csv(os.path.join(metrics_dir, 'svm_confusion_matrix.csv'), index=False)

pd.DataFrame({'y_true': y_test.values, 'y_pred': svm_pred}).to_csv(
    os.path.join(metrics_dir, 'svm_test_predictions.csv'), index=False
)

summary = {
    'best_params': svm_search.best_params_,
    'best_cv_accuracy': float(svm_search.best_score_),
    'test_accuracy': float(accuracy_score(y_test, svm_pred)),
    'n_test_samples': int(y_test.shape[0])
}
with open(os.path.join(metrics_dir, 'svm_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

joblib.dump(svm_best, os.path.join(model_dir, 'svm_best_model.joblib'))

print(f"Resultados salvos em: {metrics_dir}")
print(f"Modelo salvo em: {os.path.join(model_dir, 'svm_best_model.joblib')}")